# A reading path through the guides

The [guides](../docs/guides/reading-list.md) explain prospicio one topic at a time. Most of them reproduce a published text. This notebook is a map to the guides. It takes one small synthetic book through the stations of an actuary, from claims data to capital. At each station it runs a few lines of prospicio. It then names the guides to read next and the question that each guide answers.

The book has two lines, with money in millions of a synthetic currency:

* **Property**: seven years of ground-up claims. A known claim count and severity give the claims, so you can check the fits against the truth. This line carries the severity, the claim count, the annual aggregate, a per-risk excess of loss layer and the rate indication.
* **Casualty**: a seven by seven paid triangle, for the reserve and its uncertainty.

The stations:

1. Severity and claim count
2. The annual aggregate distribution
3. A reinsurance layer: exposure and experience rating, then a price
4. A ratemaking indication
5. Reserving: chain ladder, Mack, the ODP bootstrap
6. Risk loads, distortions and capital allocation
7. Where to go next: Bayesian models and GLMs

Every cell has a seed, and the whole notebook runs in a few seconds. To run it, install the package with `pip install prospicio jupyter` and open the notebook. To build from source instead, use `cd python && maturin develop`, which needs Rust. The notebook reads no data files. The R twin, [`reading_path.qmd`](reading_path.qmd), gives the same numbers.

In [ ]:
import math

from prospicio.aggregate import compound_moments, fft, simulate_events
from prospicio.credibility import LimitedFluctuation
from prospicio.distributions import Gamma, Grid, Lognormal, NegativeBinomial, claim_count, fit_severity
from prospicio.pricing import CollectiveModel, LossRatioExperience, Pentagon, Portfolio, Provisions, credibility_weighted
from prospicio.reinsurance import Layer, Tower
from prospicio.reserving import ChainLadder, Mack, OdpBootstrap, Triangle
from prospicio.risk import Distortion

## The book

The property claims come from a negative binomial count and a lognormal severity. The count has mean 400 and variance 600 a year. The severity has mean 0.02 and CV 3. In practice these are unknown. Here they are known, so each station can show how far its estimate is from the truth.

In [ ]:
# Money is in millions of a synthetic currency.
YEARS = list(range(2018, 2025))                                   # seven accident years
TRUE_COUNT = NegativeBinomial.from_mean_variance(400.0, 600.0)   # property claims a year
TRUE_SEVERITY = Lognormal.from_mean_cv(0.02, 3.0)                # one property claim, ground up

counts = TRUE_COUNT.sample(len(YEARS), seed=2718)
claims = TRUE_SEVERITY.sample(sum(counts), seed=6043)
by_year, start = [], 0
for n in counts:
    by_year.append(claims[start:start + n])
    start += n
for year, n, xs in zip(YEARS, counts, by_year):
    print(f"{year}: {n} claims, total {sum(xs):.2f}m, largest {max(xs):.2f}m")

## 1. Severity and claim count

The code fits a maximum likelihood lognormal to all the claims. It chooses a claim count by its dispersion. `claim_count` returns a Poisson when the variance equals the mean, and a negative binomial when the variance is larger.

In [ ]:
fit = fit_severity("lognormal", claims)
severity = fit.distribution
yearly_mean = sum(counts) / len(counts)
yearly_var = sum((n - yearly_mean) ** 2 for n in counts) / (len(counts) - 1)
count = claim_count(yearly_mean, yearly_var / yearly_mean)
print(f"severity: fitted mean {severity.mean():.4f}m, cv {severity.std() / severity.mean():.2f}"
      f" (truth {TRUE_SEVERITY.mean():.4f}m, cv 3.00)")
print(f"claim count: mean {yearly_mean:.1f}, dispersion {yearly_var / yearly_mean:.2f} -> {type(count).__name__}"
      f" (truth 400.0, dispersion 1.50)")

Seven years are enough for the severity, but not for the count: its dispersion rests on seven numbers.

**Read next.**

* [Severity and frequency](../docs/guides/severity-and-frequency.md): which severity family and which count family to use, and how to fit them, with censoring, truncation, splices and mixtures.
* [Distributions for actuaries: claim size](../docs/guides/distributions-for-actuaries-claim-size.md): limited expected values, fits to grouped, censored and truncated data, excess layers, the leveraging effect of inflation and increased limit factors (Bahnemann, CAS Monograph 2).
* [Distributions for actuaries: counts and aggregates](../docs/guides/distributions-for-actuaries-counts-aggregates.md): Poisson, mixed Poisson and negative binomial counts, and what contagion does to a count (Bahnemann, chapters 3 and 4).

## 2. The annual aggregate distribution

The total loss of the year combines the count and the severity. The severity goes on a grid of step 0.01, censored at a policy limit of 2. The FFT gives the annual distribution on that grid. The compound moments and a simulation are two checks.

In [ ]:
STEP, POINTS = 0.01, 2**12
POLICY_LIMIT = 2.0
severity_grid, _ = Grid.rounding(severity, STEP, round(POLICY_LIMIT / STEP) + 1)
annual, report = fft(count, severity_grid, POINTS)
moments = compound_moments(count, severity_grid)
simulated = simulate_events(count, severity_grid, 4000, 9157).totals().total()
print(f"FFT:        mean {annual.mean():.3f}, sd {math.sqrt(annual.variance()):.3f}, 99% {annual.quantile(0.99):.2f}, 99.5% {annual.quantile(0.995):.2f}")
print(f"moments:    mean {moments.mean:.3f}, sd {moments.mean * moments.cv:.3f}, skew {moments.skew:.3f}")
print(f"simulation: mean {simulated.mean():.3f}, 99% {simulated.quantile(0.99):.2f} (4,000 years)")

The three agree on the mean. The 99% of the simulation is within its sampling error of the FFT value.

**Read next.**

* [Aggregate distributions](../docs/guides/aggregate-distributions.md): FFT, Panjer, simulation or moments, how to size the grid, and when each is exact, fast or unsafe.
* [prospicio for aggregate users](../docs/guides/prospicio-for-aggregate-users.md): the same methods as Mildenhall's `aggregate` package, side by side, for a reader who knows that package.

## 3. A reinsurance layer

The layer is a per-risk excess of loss layer of 1.5 xs 0.5 on the property line. Exposure rating prices it from the fitted severity curve (the expected count times the layer average of the severity). Experience rating prices it from the claims that reached it (the burning cost). Only a few claims pierce 0.5. So the experience gets little credibility, and the blend is near the exposure figure.

In [ ]:
LIMIT, ATTACHMENT = 1.5, 0.5
tower = Tower([Layer("1.5 xs 0.5", limit=LIMIT, attachment=ATTACHMENT)])
layer_annual = tower.on_grid(count, severity_grid, POINTS).ceded[0]
exposure_loss = CollectiveModel(count, severity).layer_mean(LIMIT, ATTACHMENT)

in_layer = [min(max(x - ATTACHMENT, 0.0), LIMIT) for x in claims]
n_layer = sum(1 for x in claims if x > ATTACHMENT)
experience_loss = sum(in_layer) / len(YEARS)
lf = LimitedFluctuation(0.90, 0.05)
z = lf.credibility(n_layer, lf.frequency_standard())
selected = credibility_weighted(experience_loss, z, exposure_loss)
print(f"exposure (fitted curve) {exposure_loss:.4f}, on the grid {layer_annual.mean():.4f}")
print(f"experience (burning cost) {experience_loss:.4f} from {n_layer} claims, credibility {z:.3f}")
print(f"selected expected layer loss {selected:.4f}")

The price backs the layer with capital at its 99% TVaR and asks for a 10% return on that capital. The pentagon (loss, margin, premium, capital, assets) gets all five quantities from three of them:

In [ ]:
TAIL = Distortion.tvar(0.99)
pentagon = Pentagon.solve(loss=selected, assets=TAIL.measure(layer_annual), return_on_capital=0.10)
print(f"layer: loss {pentagon.loss:.4f}, margin {pentagon.margin:.4f}, premium {pentagon.premium:.4f}"
      f" ({pentagon.premium / LIMIT:.2%} rate on line), capital {pentagon.capital:.3f}")

**Read next.**

* [What a reinsurance price is](../docs/guides/reinsurance-price.md): the technical premium of an excess of loss layer and a quota share, reinstatements, and a contract alone and in a book.
* [Experience and exposure rating](../docs/guides/experience-and-exposure-rating.md): how to prepare losses as-if (development, trend, index clauses), the burning cost, exposure curves and risk profiles, and how to blend the two by credibility (after Clark).
* [Choosing a method](../docs/guides/choosing-a-method.md): which tool for which cover.

## 4. A ratemaking indication

The code applies the loss ratio method to the property line:

1. It brings earned premium to the current rate level.
2. It trends losses two years at 4% a year to the period that the new rates cover.
3. It loads losses 8% for LAE.

Property claims are short tailed, so the code takes the claims as fully developed. The indicated change compares the projected loss and LAE ratio with the permissible ratio (20% variable expense, 5% profit, 8% fixed expense). With nearly 3,000 claims, the experience is fully credible under the classical standard of 1,082 claims.

In [ ]:
earned = [12.1, 12.6, 13.0, 13.5, 13.9, 14.4, 14.8]
on_level = [1.12, 1.09, 1.07, 1.05, 1.03, 1.01, 1.00]
reported = [sum(xs) for xs in by_year]
experience = LossRatioExperience(
    earned_premium=earned, losses=reported, on_level_factor=on_level,
    development=1.0, loss_trend=1.04**2, lae_factor=1.08,
)
provisions = Provisions(variable=0.20, profit=0.05)
FIXED = 0.08
change = provisions.loss_ratio_change(experience.loss_ratio, FIXED)
z_rate = lf.credibility(sum(counts), lf.frequency_standard())
print("projected loss and LAE ratio by year", " ".join(f"{r:.1%}" for r in experience.loss_ratio_by_year))
print(f"all years {experience.loss_ratio:.1%}; permissible {provisions.permissible_loss_ratio(FIXED):.1%}")
print(f"indicated change {change:+.1%}, credibility {z_rate:.2f} ({sum(counts):,} claims)")

**Read next.**

* [Ratemaking with Werner and Modlin](../docs/guides/ratemaking-werner-modlin.md): each step of the overall indication (premium at the current rate level by the parallelogram method, trend, development, expenses and the permissible loss ratio, credibility and its complements).
* [Werner and Modlin's indications](../docs/guides/ratemaking-werner-modlin-indications.md): the three worked indications of the book end to end, and increased limits and deductibles.

## 5. Reserving

The casualty line has a paid triangle. The chain ladder gives the ultimate by accident year. Mack's formula gives the standard error of the reserve without a distributional assumption. The ODP bootstrap gives a distribution of the reserve.

In [ ]:
paid = [  # casualty, cumulative paid by accident year, ages 12, 24, ... months
    [2.41, 4.22, 5.27, 5.90, 6.20, 6.32, 6.40],
    [2.66, 5.19, 7.16, 8.59, 9.45, 9.92],
    [2.52, 4.54, 5.54, 6.04, 6.40],
    [2.90, 5.86, 7.91, 9.18],
    [3.05, 5.18, 6.73],
    [3.21, 6.03],
    [3.40],
]
tri = Triangle.from_long(
    [y for y, row in zip(YEARS, paid) for _ in row],
    [12 * (k + 1) for row in paid for k in range(len(row))],
    {"paid": [v for row in paid for v in row]},
)
cl = ChainLadder().fit(tri, "paid")
print("age-to-age", " ".join(f"{f:.3f}" for f in cl.ldf))
print("ultimate", " ".join(f"{u:.2f}" for u in cl.ultimate))
print(f"chain ladder reserve {cl.total_reserve:.2f}m")

In [ ]:
mack = Mack().fit(tri, "paid")
boot = OdpBootstrap(n_sims=2000, seed=4471).fit(tri, "paid").reserves.total()
print(f"Mack: reserve {mack.total_reserve:.2f}m, standard error {mack.total_standard_error:.2f}m, cv {mack.total_cv:.3f}")
print(f"ODP bootstrap: mean {boot.mean():.2f}m, sd {math.sqrt(boot.variance()):.2f}m, 99.5% {boot.quantile(0.995):.2f}m (2,000 simulations)")

Mack and the bootstrap give a similar result: a reserve of about 15 with a CV near 13%.

**Read next.**

* [Reserving methods](../docs/guides/reserving-methods.md): which method to use when (chain ladder, Bornhuetter-Ferguson, Cape Cod, Mack, the bootstraps, tails and the one-year view), with the uncertainty of the reserve as a distribution.
* [Reserving with Friedland's techniques](../docs/guides/reserving-friedland.md): development, expected claims, Bornhuetter-Ferguson, Benktander and Cape Cod on the data of the CAS text itself.
* [Taylor and McGuire's ODP GLM](../docs/guides/reserving-taylor-mcguire.md): the chain ladder as an over-dispersed Poisson GLM, its prediction error and the bootstrap, and the reduced models (CAS Monograph 3).
* [Excess-layer loss development](../docs/guides/reserving-excess-layers.md): layer factors derived from the dense ground-up triangle and a claim size distribution, blended with the data of the layer itself by credibility, and the layer share as a Bayesian Bornhuetter-Ferguson prior.

## 6. Risk loads, distortions and capital allocation

The two lines go into one book as independent units on the same grid. The property unit is the annual loss from station 2. The casualty unit is the reserve risk from station 5, a gamma with the mean and CV from Mack. Assets at the 99% VaR of the book back it, and these assets are less than the sum of the two VaRs. The code calibrates each of five distortion families to a 10% return on the capital of the book. So all five charge the same total margin, and the natural allocation splits that margin between the lines.

In [ ]:
reserve_risk = Gamma.from_mean_cv(mack.total_reserve, mack.total_cv)
reserve_grid, _ = Grid.rounding(reserve_risk, STEP, POINTS)
book = Portfolio.from_independent(["property", "casualty reserve"], [annual, reserve_grid])
alone = annual.quantile(0.99) + reserve_grid.quantile(0.99)
print(f"99% VaR: units alone {alone:.2f}m, together {book.assets(0.99):.2f}m")
FAMILIES = ["ccoc", "ph", "wang", "dual", "tvar"]
print(f"{'distortion':>10} {'property':>9} {'reserve':>9} {'total':>8}")
for family in FAMILIES:
    d = book.calibrate(family, p=0.99, return_on_capital=0.10)
    priced = book.price(d, p=0.99)
    margins = [u.margin for u in priced.allocated]
    print(f"{family:>10} {margins[0]:9.3f} {margins[1]:9.3f} {priced.total.margin:8.3f}")

Here the five distortions split the margin almost the same way, because neither line has a catastrophe tail. They differ when one unit is much more skewed than the other, as the case studies show.

**Read next.**

* [Risk loads and distortions](../docs/guides/risk-loads-and-distortions.md): which risk load or distortion to use, how to calibrate it to quotes, and how to price a contract in a portfolio.
* [Pricing Insurance Risk: risk measures](../docs/guides/pir-risk-measures.md): VaR, TVaR and EPD, their properties, and a distortion as a spectral risk measure (Mildenhall and Major, Part I).
* [Pricing Insurance Risk: portfolio pricing](../docs/guides/pir-portfolio-pricing.md): the constant cost of capital price, classical premium principles and the calibrated distortions (Part II).
* [Pricing Insurance Risk: price allocation](../docs/guides/pir-price-allocation.md) and [allocation in practice](../docs/guides/pir-price-allocation-practice.md): the classical allocations against the natural allocation, and which question each answers (Part III).
* [Pricing Insurance Risk: the case studies](../docs/guides/pricing-insurance-risk-case-studies.md): books with a catastrophe unit, gross and net, where the distortions disagree by unit.
* [CAS Monograph 15](../docs/guides/capital-modeling-monograph-15.md) and its chapter guides on [theory](../docs/guides/capital-modeling-monograph-15-theory.md), [applications](../docs/guides/capital-modeling-monograph-15-applications.md), [practice](../docs/guides/capital-modeling-monograph-15-practice.md) and [dependence by Iman-Conover](../docs/guides/capital-modeling-monograph-15-iman-conover.md): capital modeling and portfolio management (Major and Mildenhall), from risk measures to reinsurance decisions and correlated units.

## 7. Where to go next

The stations above use closed forms, grids and quick simulations. Two families of guides go further, with fits that take longer than this notebook should:

* **Bayesian reserving.** [Meyers' models](../docs/guides/reserving-meyers.md) (CAS Monograph 8) go from Mack and the ODP bootstrap to the Bayesian CRC, CSR and CAY models. They test the models by the percentile of the outcome. [Compartmental reserving](../docs/guides/reserving-compartmental.md) (Gesmann and Morris) models paid and outstanding together, with reporting and payment rates.
* **GLMs for rating.** [GLMs for insurance rating](../docs/guides/glm-ratemaking-goldburd.md) and [their validation](../docs/guides/glm-ratemaking-goldburd-validation.md) (CAS Monograph 5) build a multiplicative rating plan and test it. [From GLMs to comprehensive pricing](../docs/guides/glm-to-comprehensive-pricing-chalk.md) and [its second part](../docs/guides/glm-to-comprehensive-pricing-chalk-2.md) (CAS Monograph 16) add penalized GLMs, GAMs, offsets and lessons from boosting models.

The [reading list](../docs/guides/reading-list.md) maps each source text to its guides. The other notebooks here are worked analyses that use these pieces together.